# LangChain y LangSmith · Modelos, prompts y cadenas

Las piezas básicas de [LangChain](https://docs.langchain.com/) 1.x: mensajes, modelos de chat, plantillas de prompt, analizadores de salida y la composición de *runnables* con el operador `|` (LCEL).

Para que el notebook se ejecute sin claves de API ni conexión, los modelos son locales y deterministas (`GenericFakeChatModel` o un modelo propio basado en reglas). Con un modelo real el código es el mismo: basta cambiar la línea que crea el modelo, por ejemplo `init_chat_model("anthropic:claude-sonnet-5-5")` con `langchain-anthropic` instalado y `ANTHROPIC_API_KEY` definida.

## 1. Mensajes

Crear una conversación con un mensaje de sistema, dos del usuario y una respuesta del asistente usando las clases `SystemMessage`, `HumanMessage` y `AIMessage`. Construir la misma conversación con diccionarios `{"role", "content"}`, convertirla con `convert_to_messages` y comprobar que ambas son iguales. Mostrar la conversación con `pretty_print` y contar los mensajes de cada tipo.

In [1]:
from collections import Counter

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, convert_to_messages

conversacion = [
    SystemMessage("Eres un guía turístico de Canarias."),
    HumanMessage("¿Cuál es la isla más grande?"),
    AIMessage("Tenerife, con 2034 km²."),
    HumanMessage("¿Y la más pequeña habitada?"),
]
como_dicts = [
    {"role": "system", "content": "Eres un guía turístico de Canarias."},
    {"role": "user", "content": "¿Cuál es la isla más grande?"},
    {"role": "assistant", "content": "Tenerife, con 2034 km²."},
    {"role": "user", "content": "¿Y la más pequeña habitada?"},
]
convertida = convert_to_messages(como_dicts)

assert [(m.type, m.content) for m in convertida] == [(m.type, m.content) for m in conversacion]
for mensaje in conversacion:
    mensaje.pretty_print()
print(Counter(m.type for m in conversacion))

================================ System Message ================================

Eres un guía turístico de Canarias.
================================ Human Message =================================

¿Cuál es la isla más grande?
================================== Ai Message ==================================

Tenerife, con 2034 km².
================================ Human Message =================================

¿Y la más pequeña habitada?
Counter({'human': 2, 'system': 1, 'ai': 1})


## 2. Un modelo de prueba

Crear un `GenericFakeChatModel` que devuelva dos respuestas fijas. Llamarlo con `invoke` y con `stream`, mostrando cada fragmento (*chunk*) que llega y comprobando que, sumados, reconstruyen el mensaje completo.

In [2]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

modelo = GenericFakeChatModel(messages=iter([
    AIMessage("La Graciosa, con unos 29 km²."),
    AIMessage("El Teide mide 3715 metros y es el pico más alto de España."),
]))

respuesta = modelo.invoke("¿Cuál es la isla habitada más pequeña?")
print(type(respuesta).__name__, "->", respuesta.content)

trozos = []
for chunk in modelo.stream("¿Cuánto mide el Teide?"):
    trozos.append(chunk)
    print(repr(chunk.content), end=" | ")
completo = sum(trozos[1:], trozos[0])
print("\n" + completo.content)
assert completo.content == "El Teide mide 3715 metros y es el pico más alto de España."

AIMessage -> La Graciosa, con unos 29 km².
'El' | ' ' | 'Teide' | ' ' | 'mide' | ' ' | '3715' | ' ' | 'metros' | ' ' | 'y' | ' ' | 'es' | ' ' | 'el' | ' ' | 'pico' | ' ' | 'más' | ' ' | 'alto' | ' ' | 'de' | ' ' | 'España.' | 
El Teide mide 3715 metros y es el pico más alto de España.


## 3. Un modelo de chat propio

Crear `ModeloReglas`, una subclase de `BaseChatModel` que recibe una lista de reglas `(patrón, plantilla)`: responde con la plantilla de la primera expresión regular que encaje en el último mensaje (las plantillas pueden usar los grupos, como `\1`) y, si no encaja ninguna, con una respuesta por defecto. Debe contar cuántas veces se ha llamado. Probarlo con `invoke` y `batch`.

In [3]:
import re

from langchain_core.language_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult
from pydantic import PrivateAttr


class ModeloReglas(BaseChatModel):
    reglas: list[tuple[str, str]] = []
    por_defecto: str = "No lo sé."
    _llamadas: int = PrivateAttr(default=0)

    @property
    def llamadas(self) -> int:
        return self._llamadas

    @property
    def _llm_type(self) -> str:
        return "reglas"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self._llamadas += 1
        texto = messages[-1].text
        respuesta = self.por_defecto
        for patron, plantilla in self.reglas:
            if m := re.search(patron, texto, re.IGNORECASE | re.DOTALL):
                respuesta = m.expand(plantilla)
                break
        return ChatResult(generations=[ChatGeneration(message=AIMessage(respuesta))])


modelo = ModeloReglas(reglas=[
    (r"hola|buenas", "¡Buenas! ¿En qué puedo ayudarles?"),
    (r"capital de (\w+)", r"No tengo datos de la capital de \1."),
    (r"traduce '(.+)' al inglés", r"Traducción de '\1': pendiente."),
])
print(modelo.invoke("Hola, ¿qué tal?").content)
for r in modelo.batch(["¿Cuál es la capital de Portugal?", "Traduce 'papa arrugada' al inglés", "¿Qué hora es?"]):
    print(r.content)
assert modelo.llamadas == 4

¡Buenas! ¿En qué puedo ayudarles?
No tengo datos de la capital de Portugal.
Traducción de 'papa arrugada': pendiente.
No lo sé.


## 4. Plantillas de prompt

Crear un `ChatPromptTemplate` con un mensaje de sistema que tenga la variable `{idioma}`, un `MessagesPlaceholder` para el historial y un mensaje humano con `{pregunta}`. Fijar el idioma con `partial` y mostrar los mensajes que genera. Crear también un prompt *few-shot* con `FewShotChatMessagePromptTemplate` que enseñe a convertir palabras al plural con dos ejemplos.

In [4]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde siempre en {idioma} y en una sola frase."),
    MessagesPlaceholder("historial"),
    ("human", "{pregunta}"),
])
print(prompt.input_variables)
prompt_es = prompt.partial(idioma="español")
valor = prompt_es.invoke({
    "historial": [HumanMessage("Me llamo Ana."), AIMessage("Encantado, Ana.")],
    "pregunta": "¿Cómo me llamo?",
})
for m in valor.to_messages():
    print(f"{m.type:>6}: {m.content}")

ejemplo = ChatPromptTemplate.from_messages([("human", "{palabra}"), ("ai", "{plural}")])
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=ejemplo,
    examples=[{"palabra": "guagua", "plural": "guaguas"}, {"palabra": "papel", "plural": "papeles"}],
)
prompt_plural = ChatPromptTemplate.from_messages([
    ("system", "Devuelve el plural de la palabra."), few_shot, ("human", "{palabra}"),
])
print(prompt_plural.invoke({"palabra": "lápiz"}).to_string())

['historial', 'idioma', 'pregunta']
system: Responde siempre en español y en una sola frase.
 human: Me llamo Ana.
    ai: Encantado, Ana.
 human: ¿Cómo me llamo?
System: Devuelve el plural de la palabra.
Human: guagua
AI: guaguas
Human: papel
AI: papeles
Human: lápiz


## 5. Primera cadena

Componer `prompt | modelo | StrOutputParser()` con el modelo de reglas para un traductor de palabras sueltas al inglés (un pequeño diccionario convertido en reglas). Mostrar los esquemas de entrada y salida de la cadena, traducir varias palabras con `batch` y comprobar que la salida es una cadena de texto.

In [5]:
from langchain_core.output_parsers import StrOutputParser

diccionario = {"guagua": "bus", "papa": "potato", "millo": "corn", "baifo": "goat kid"}
traductor = ModeloReglas(
    reglas=[(rf"palabra: {es}\b", en) for es, en in diccionario.items()],
    por_defecto="(sin traducción)",
)
prompt_traduccion = ChatPromptTemplate.from_messages([
    ("system", "Traduce al inglés la palabra que te den. Devuelve solo la traducción."),
    ("human", "Palabra: {palabra}"),
])
cadena = prompt_traduccion | traductor | StrOutputParser()

print(cadena.input_schema.model_json_schema())
print(cadena.output_schema.model_json_schema())
resultados = cadena.batch([{"palabra": p} for p in ["guagua", "millo", "baifo", "gofio"]])
print(resultados)
assert all(isinstance(r, str) for r in resultados) and resultados[0] == "bus"

{'properties': {'palabra': {'title': 'Palabra', 'type': 'string'}}, 'required': ['palabra'], 'title': 'PromptInput', 'type': 'object'}
{'title': 'StrOutputParserOutput', 'type': 'string'}
['bus', 'corn', 'goat kid', '(sin traducción)']


## 6. Runnables en paralelo

Con `RunnableLambda`, `RunnableParallel` y `RunnablePassthrough.assign`, crear una cadena que reciba `{"texto": ...}` y devuelva, calculados en paralelo, el número de palabras, la palabra más larga y la frecuencia de cada vocal; después añadir un campo `resumen` que dependa de esos resultados. No hace falta ningún modelo: los runnables también componen funciones normales.

In [6]:
from operator import itemgetter

from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

palabras = itemgetter("texto") | RunnableLambda(lambda t: re.findall(r"\w+", t))
analisis = RunnablePassthrough.assign(
    estadisticas=RunnableParallel(
        n_palabras=palabras | len,
        mas_larga=palabras | RunnableLambda(lambda ps: max(ps, key=len)),
        vocales=itemgetter("texto") | RunnableLambda(lambda t: {v: t.lower().count(v) for v in "aeiou"}),
    )
).assign(resumen=lambda d: f"{d['estadisticas']['n_palabras']} palabras; la más larga es '{d['estadisticas']['mas_larga']}'")

resultado = analisis.invoke({"texto": "En un lugar de Lanzarote, de cuyo nombre no quiero acordarme"})
print(resultado)
assert resultado["estadisticas"]["n_palabras"] == 11

{'texto': 'En un lugar de Lanzarote, de cuyo nombre no quiero acordarme', 'estadisticas': {'n_palabras': 11, 'mas_larga': 'Lanzarote', 'vocales': {'a': 5, 'e': 7, 'i': 1, 'o': 6, 'u': 4}}, 'resumen': "11 palabras; la más larga es 'Lanzarote'"}


## 7. Enrutar entre cadenas

Crear un clasificador (modelo de reglas) que etiquete cada pregunta como `matematicas`, `geografia` u `otro`, y una cadena distinta para cada etiqueta. Enrutar de dos formas: con `RunnableBranch` y con una función que devuelve el runnable que toca. Comprobar que ambas dan el mismo resultado.

In [7]:
from langchain_core.runnables import RunnableBranch

clasificador = (
    ChatPromptTemplate.from_template("Clasifica la pregunta: {pregunta}")
    | ModeloReglas(reglas=[(r"\d+\s*[-+*/]\s*\d+|cuánto es", "matematicas"), (r"isla|capital|río|montaña", "geografia")], por_defecto="otro")
    | StrOutputParser()
)


def calcular(d):
    expresion = re.search(r"\d+(?:\s*[-+*/]\s*\d+)+", d["pregunta"]).group()
    return f"{expresion} = {eval(expresion)}"  # la regex solo deja pasar dígitos y operadores


cadenas = {
    "matematicas": RunnableLambda(calcular),
    "geografia": RunnableLambda(lambda d: f"Pregunta de geografía recibida: {d['pregunta']}"),
    "otro": RunnableLambda(lambda d: "Solo sé de matemáticas y geografía."),
}
con_tema = RunnablePassthrough.assign(tema=clasificador)

rama = con_tema | RunnableBranch(
    (lambda d: d["tema"] == "matematicas", cadenas["matematicas"]),
    (lambda d: d["tema"] == "geografia", cadenas["geografia"]),
    cadenas["otro"],
)
funcion = con_tema | RunnableLambda(lambda d: cadenas[d["tema"]])

preguntas = [{"pregunta": p} for p in ["¿Cuánto es 12 * 7 + 1?", "¿Cuál es la isla más oriental?", "¿Te gusta el queso?"]]
for p, a, b in zip(preguntas, rama.batch(preguntas), funcion.batch(preguntas)):
    print(f"{p['pregunta']:<35} -> {a}")
    assert a == b

¿Cuánto es 12 * 7 + 1?              -> 12 * 7 + 1 = 85
¿Cuál es la isla más oriental?      -> Pregunta de geografía recibida: ¿Cuál es la isla más oriental?
¿Te gusta el queso?                 -> Solo sé de matemáticas y geografía.


## 8. Salida estructurada

Definir un modelo de Pydantic `Receta` (nombre, raciones, ingredientes) y pedirle al modelo que responda en JSON siguiendo las instrucciones de formato de `PydanticOutputParser`. Mostrar esas instrucciones, analizar la respuesta en un objeto `Receta` y comprobar que una respuesta mal formada lanza `OutputParserException` (capturarla e imprimirla).

In [8]:
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class Receta(BaseModel):
    nombre: str = Field(description="Nombre del plato")
    raciones: int = Field(description="Número de raciones", gt=0)
    ingredientes: list[str] = Field(description="Ingredientes sin cantidades")


parser = PydanticOutputParser(pydantic_object=Receta)
print(parser.get_format_instructions()[:300], "...")

prompt_receta = ChatPromptTemplate.from_messages([
    ("system", "Extrae la receta.\n{formato}"),
    ("human", "{texto}"),
]).partial(formato=parser.get_format_instructions())
cocinero = ModeloReglas(
    reglas=[(r"papas arrugadas", '{"nombre": "Papas arrugadas", "raciones": 4, "ingredientes": ["papas", "sal gorda", "agua"]}')],
    por_defecto='{"nombre": "Algo", "raciones": "muchas"}',
)
cadena_receta = prompt_receta | cocinero | parser

receta = cadena_receta.invoke({"texto": "Para cuatro personas: papas arrugadas con sal gorda y agua."})
print(repr(receta))
assert receta.raciones == 4

try:
    cadena_receta.invoke({"texto": "Un plato cualquiera"})
except OutputParserException as e:
    print("Error de análisis:", str(e).splitlines()[0])

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} ...
Receta(nombre='Papas arrugadas', raciones=4, ingredientes=['papas', 'sal gorda', 'agua'])
Error de análisis: Failed to parse Receta from completion {"nombre": "Algo", "raciones": "muchas"}. Got: 2 validation errors for Receta


## 9. Reintentos y alternativas

Simular un modelo inestable que falla las dos primeras llamadas con `ConnectionError`. Envolverlo con `with_retry` para que acabe respondiendo y contar los intentos. Después, con otro modelo que siempre falla, usar `with_fallbacks` para pasar a un modelo de respaldo.

In [9]:
intentos = 0


def inestable(entrada):
    global intentos
    intentos += 1
    if intentos <= 2:
        raise ConnectionError(f"timeout en el intento {intentos}")
    return f"Respuesta tras {intentos} intentos"


con_reintentos = RunnableLambda(inestable).with_retry(
    retry_if_exception_type=(ConnectionError,), stop_after_attempt=3, wait_exponential_jitter=False
)
print(con_reintentos.invoke("hola"))
assert intentos == 3


def caido(entrada):
    raise ConnectionError("servicio caído")


respaldo = ModeloReglas(por_defecto="Respuesta del modelo de respaldo")
con_alternativa = RunnableLambda(caido).with_fallbacks([respaldo])
print(con_alternativa.invoke("hola").content)

Respuesta tras 3 intentos
Respuesta del modelo de respaldo


## 10. Callbacks y eventos

Crear un `BaseCallbackHandler` que registre el inicio y el fin de cada cadena y de cada llamada al modelo, y pasarlo en la configuración (`config={"callbacks": [...]}`) junto con etiquetas y un `run_name`. Después recorrer los eventos de la misma cadena con `astream_events` y contar cuántos hay de cada tipo.

In [10]:
from langchain_core.callbacks import BaseCallbackHandler


class Registro(BaseCallbackHandler):
    def __init__(self):
        self.eventos = []

    def on_chain_start(self, serialized, inputs, *, run_id, parent_run_id=None, **kwargs):
        self.eventos.append(("inicio cadena", kwargs.get("name"), parent_run_id is None))

    def on_chain_end(self, outputs, *, run_id, **kwargs):
        self.eventos.append(("fin cadena", None, None))

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self.eventos.append(("inicio modelo", len(messages[0]), kwargs.get("tags")))

    def on_llm_end(self, response, *, run_id, **kwargs):
        self.eventos.append(("fin modelo", response.generations[0][0].text, None))


registro = Registro()
cadena.invoke({"palabra": "papa"}, config={"callbacks": [registro], "tags": ["traductor"], "run_name": "traducir"})
for evento in registro.eventos:
    print(evento)

eventos = Counter()
async for evento in cadena.astream_events({"palabra": "millo"}):
    eventos[evento["event"]] += 1
print(eventos)

('inicio cadena', 'traducir', True)
('inicio cadena', 'ChatPromptTemplate', False)
('fin cadena', None, None)
('inicio modelo', 2, ['seq:step:2', 'traductor'])
('fin modelo', 'potato', None)
('inicio cadena', 'StrOutputParser', False)
('fin cadena', None, None)
('fin cadena', None, None)
Counter({'on_chain_start': 1, 'on_prompt_start': 1, 'on_prompt_end': 1, 'on_chat_model_start': 1, 'on_chat_model_end': 1, 'on_parser_start': 1, 'on_parser_stream': 1, 'on_chain_stream': 1, 'on_parser_end': 1, 'on_chain_end': 1})


## 11. Memoria de conversación

Mantener el historial de una conversación en una lista y pasarlo al prompt con `MessagesPlaceholder`. El modelo de reglas debe recordar el nombre del usuario si aparece en el historial. Antes de cada llamada, recortar el historial con `trim_messages` para quedarse con los últimos 4 mensajes (contando mensajes en lugar de tokens) sin perder el de sistema y empezando por un mensaje humano.

In [11]:
from langchain_core.messages import trim_messages


class ModeloMemoria(ModeloReglas):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        if not re.search(r"cómo me llamo", messages[-1].text, re.IGNORECASE):
            return super()._generate(messages, stop, run_manager, **kwargs)
        nombres = [m.group(1) for msg in messages[:-1] if (m := re.search(r"me llamo (\w+)", msg.text, re.IGNORECASE))]
        texto = f"Se llama {nombres[-1]}." if nombres else "No me lo ha dicho."
        return ChatResult(generations=[ChatGeneration(message=AIMessage(texto))])


recortar = trim_messages(max_tokens=5, token_counter=len, strategy="last", include_system=True, start_on="human")
chat = ChatPromptTemplate.from_messages([MessagesPlaceholder("mensajes")]) | ModeloMemoria(
    reglas=[(r"me llamo (\w+)", r"Encantado, \1.")], por_defecto="Entendido."
)

historial = [SystemMessage("Eres un asistente amable.")]
for frase in ["Hola, me llamo Yaiza.", "¿Cómo me llamo?", "Hace sol.", "Me gusta el mar.", "Voy a la playa.", "¿Cómo me llamo?"]:
    historial.append(HumanMessage(frase))
    recortado = recortar.invoke(historial)
    respuesta = chat.invoke({"mensajes": recortado})
    historial.append(respuesta)
    print(f"{frase:<25} -> {respuesta.content:<20} (ve {len(recortado)} mensajes)")
assert historial[4].content == "Se llama Yaiza." and historial[-1].content == "No me lo ha dicho."

Hola, me llamo Yaiza.     -> Encantado, Yaiza.    (ve 2 mensajes)
¿Cómo me llamo?           -> Se llama Yaiza.      (ve 4 mensajes)
Hace sol.                 -> Entendido.           (ve 4 mensajes)
Me gusta el mar.          -> Entendido.           (ve 4 mensajes)
Voy a la playa.           -> Entendido.           (ve 4 mensajes)
¿Cómo me llamo?           -> No me lo ha dicho.   (ve 4 mensajes)


## 12. Caché de respuestas

Activar una caché en memoria con `set_llm_cache(InMemoryCache())` y comprobar, con el contador de llamadas del modelo de reglas, que repetir la misma pregunta no vuelve a llamar al modelo. Medir el tiempo de la primera y la segunda llamada simulando un modelo lento.

In [12]:
import time

from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache


class ModeloLento(ModeloReglas):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        time.sleep(0.3)
        return super()._generate(messages, stop, run_manager, **kwargs)


set_llm_cache(InMemoryCache())
lento = ModeloLento(por_defecto="Respuesta costosa")
for i in range(3):
    inicio = time.perf_counter()
    lento.invoke("¿Cuál es el sentido de la vida?")
    print(f"Llamada {i + 1}: {time.perf_counter() - inicio:.3f} s")
lento.invoke("Otra pregunta distinta")
print("Llamadas reales al modelo:", lento.llamadas)
assert lento.llamadas == 2
set_llm_cache(None)

Llamada 1: 0.301 s
Llamada 2: 0.001 s
Llamada 3: 0.000 s


Llamadas reales al modelo: 2
